# Geospatial Pipeline

Este notebook construye las relaciones espaciales entre los cuatro datasets:
1. Asigna cada establecimiento IPRESS a su distrito (por UBIGEO)
2. Asigna cada centro poblado a su distrito (por UBIGEO directo + spatial join para los sin código)
3. Genera los GeoDataFrames base para el análisis de acceso


In [1]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
import sys

sys.path.append(str(Path("..").resolve()))
from utils import ubigeo_to_str

ROOT = Path("..").resolve()
PROCESSED_DIR = ROOT / "data" / "processed"

# Cargamos los datasets ya limpios
gdf_distritos = gpd.read_file(PROCESSED_DIR / "distritos_clean.gpkg")
df_ipress     = pd.read_csv(PROCESSED_DIR / "ipress_clean.csv", dtype={"ubigeo": str})
gdf_cp        = gpd.read_file(PROCESSED_DIR / "centros_poblados_clean.gpkg")

print("Distritos :", gdf_distritos.shape)
print("IPRESS    :", df_ipress.shape)
print("C. Poblados:", gdf_cp.shape)


Distritos : (1873, 6)
IPRESS    : (20790, 12)
C. Poblados: (135942, 6)


## 1. IPRESS con geometría

Creamos un GeoDataFrame de IPRESS usando las columnas `norte` (longitud) y `este` (latitud).
Solo los 7,953 establecimientos con coordenadas válidas tendrán geometría.
Los demás se vincularán a distritos únicamente por UBIGEO.


In [2]:
# Separamos los que tienen coordenadas válidas de los que no
ipress_con_coords = df_ipress.dropna(subset=["norte", "este"]).copy()
ipress_sin_coords = df_ipress[df_ipress["norte"].isna()].copy()

# Creamos geometría: norte = longitud (X), este = latitud (Y)
gdf_ipress = gpd.GeoDataFrame(
    ipress_con_coords,
    geometry=gpd.points_from_xy(ipress_con_coords["norte"], ipress_con_coords["este"]),
    crs="EPSG:4326"
)

print(f"IPRESS con geometría : {len(gdf_ipress):,}")
print(f"IPRESS sin geometría : {len(ipress_sin_coords):,}")
print(f"CRS: {gdf_ipress.crs}")
print(f"\nMuestra:")
print(gdf_ipress[["codigo_unico", "nombre_del_establecimiento", "ubigeo", "geometry"]].head(3).to_string())


IPRESS con geometría : 7,941
IPRESS sin geometría : 12,849
CRS: EPSG:4326

Muestra:
   codigo_unico         nombre_del_establecimiento  ubigeo                    geometry
1          7050                             AMBATO  060611  POINT (-78.85838 -6.13352)
2            99          SANTA ISABEL DE YUMBATURO  160302  POINT (-74.25814 -4.58151)
6          7278  PUESTO DE SALUD HEROES DEL CENEPA  150142  POINT (-76.93061 -12.2487)


## 2. Asignar IPRESS a distritos

Usamos UBIGEO como clave de unión directa — es más confiable que un spatial join
porque el UBIGEO ya viene en el dato original y no depende de la precisión de las coordenadas.


In [3]:
# Unimos todos los IPRESS (con y sin coords) al shapefile de distritos por UBIGEO
df_ipress_dist = df_ipress.merge(
    gdf_distritos[["ubigeo", "departamen", "provincia", "distrito"]],
    on="ubigeo",
    how="left",
    suffixes=("", "_dist")
)

# Cuántos IPRESS quedaron sin distrito (UBIGEO no encontrado en shapefile)
sin_match = df_ipress_dist["distrito"].isna().sum()
print(f"IPRESS con distrito asignado : {len(df_ipress_dist) - sin_match:,}")
print(f"IPRESS sin match en distritos: {sin_match:,}")

# Resumen de establecimientos por departamento
print("\nTop 10 departamentos con más IPRESS:")
print(df_ipress_dist["departamen"].value_counts().head(10))


IPRESS con distrito asignado : 20,790
IPRESS sin match en distritos: 0

Top 10 departamentos con más IPRESS:
departamen
LIMA           7415
CAJAMARCA      1097
PIURA          1055
AREQUIPA       1003
JUNIN           900
CUSCO           861
CALLAO          768
LA LIBERTAD     677
LAMBAYEQUE      673
PUNO            612
Name: count, dtype: int64


## 3. Asignar Centros Poblados a distritos

Para los centros poblados con UBIGEO directo (47%): join por UBIGEO.
Para los sin UBIGEO (53%): spatial join — asignamos cada punto al polígono distrital que lo contiene.


In [4]:
# Separamos los que ya tienen UBIGEO de los que no
cp_con_ubigeo = gdf_cp[gdf_cp["ubigeo"].notna()].copy()
cp_sin_ubigeo = gdf_cp[gdf_cp["ubigeo"].isna()].copy()

print(f"CP con UBIGEO directo: {len(cp_con_ubigeo):,}")
print(f"CP sin UBIGEO (spatial join): {len(cp_sin_ubigeo):,}")

# Spatial join: asignamos cada punto al distrito que lo contiene
cp_joined = gpd.sjoin(
    cp_sin_ubigeo[["nom_poblad", "categoria", "x", "y", "geometry"]],
    gdf_distritos[["ubigeo", "departamen", "provincia", "distrito", "geometry"]],
    how="left",
    predicate="within"
)

# Limpiamos columnas del join
cp_joined = cp_joined.drop(columns=["index_right"])

print(f"\nCP asignados por spatial join: {cp_joined['ubigeo'].notna().sum():,}")
print(f"CP sin asignar (fuera de polígonos): {cp_joined['ubigeo'].isna().sum():,}")


CP con UBIGEO directo: 64,196
CP sin UBIGEO (spatial join): 71,746

CP asignados por spatial join: 71,528
CP sin asignar (fuera de polígonos): 218


In [5]:
# Agregamos las columnas de distrito a los CP que ya tenían UBIGEO
cp_con_ubigeo = cp_con_ubigeo.merge(
    gdf_distritos[["ubigeo", "departamen", "provincia", "distrito"]],
    on="ubigeo",
    how="left"
)

# Unificamos columnas para que ambos grupos tengan la misma estructura
columnas_finales = ["nom_poblad", "categoria", "ubigeo", "departamen", "provincia", "distrito", "x", "y", "geometry"]

cp_joined_final = cp_joined.reindex(columns=columnas_finales)
cp_con_ubigeo_final = cp_con_ubigeo.reindex(columns=columnas_finales)

# Concatenamos ambos grupos
gdf_cp_final = pd.concat([cp_con_ubigeo_final, cp_joined_final], ignore_index=True)

print(f"Total centros poblados con distrito: {gdf_cp_final['ubigeo'].notna().sum():,}")
print(f"Sin distrito asignado              : {gdf_cp_final['ubigeo'].isna().sum():,}")
print(f"Total                              : {len(gdf_cp_final):,}")


Total centros poblados con distrito: 135,724
Sin distrito asignado              : 218
Total                              : 135,942


In [6]:
# Convertimos a GeoDataFrame antes de guardar
gdf_cp_final = gpd.GeoDataFrame(gdf_cp_final, geometry="geometry", crs="EPSG:4326")

# Guardamos
output_path = PROCESSED_DIR / "centros_poblados_distritales.gpkg"
gdf_cp_final.to_file(output_path, driver="GPKG")
print(f"Guardado en: {output_path}")


Guardado en: C:\Users\esarmiento\Documents\GitHub\emergency_access_peru\data\processed\centros_poblados_distritales.gpkg


## 4. Agregación a nivel distrito

Con IPRESS y centros poblados asignados a distritos, construimos las métricas base:
- Número de IPRESS por distrito
- IPRESS por categoría (para distinguir hospitales de postas)
- Número de centros poblados por distrito


In [8]:
# Conteo de IPRESS por distrito
ipress_por_dist = df_ipress.groupby("ubigeo").agg(
    n_ipress=("codigo_unico", "count"),
    n_ipress_con_coords=("norte", lambda x: x.notna().sum()),
    n_camas=("camas", "sum")
).reset_index()

# Extraemos el nivel correctamente: parte antes del guión (I, II, III, Sin)
df_ipress["nivel"] = df_ipress["categoria"].str.split("-").str[0]

nivel_pivot = df_ipress.groupby(["ubigeo", "nivel"]).size().unstack(fill_value=0).reset_index()
nivel_pivot.columns.name = None

print("Niveles disponibles:", sorted(df_ipress["nivel"].unique()))
print("\nNivel pivot (muestra):")
print(nivel_pivot.head(3).to_string())



Niveles disponibles: ['I', 'II', 'III', 'Sin Categoría']

Nivel pivot (muestra):
   ubigeo   I  II  III  Sin Categoría
0  010101  16   3    0             17
1  010102   1   0    0              0
2  010103   3   0    0              0


In [9]:
# Conteo de centros poblados por distrito
cp_por_dist = gdf_cp_final.groupby("ubigeo").agg(
    n_centros_poblados=("nom_poblad", "count")
).reset_index()

print("Centros poblados por distrito (muestra):")
print(cp_por_dist.head(5).to_string())
print(f"\nDistritos con centros poblados: {len(cp_por_dist):,}")


Centros poblados por distrito (muestra):
   ubigeo  n_centros_poblados
0  010101                  48
1  010102                  10
2  010103                  27
3  010104                  21
4  010105                  48

Distritos con centros poblados: 1,913


In [10]:
# Unimos todas las métricas base al GeoDataFrame de distritos
gdf_base = gdf_distritos.copy()

gdf_base = gdf_base.merge(ipress_por_dist, on="ubigeo", how="left")
gdf_base = gdf_base.merge(nivel_pivot, on="ubigeo", how="left")
gdf_base = gdf_base.merge(cp_por_dist, on="ubigeo", how="left")

# Rellenamos con 0 los distritos sin establecimientos ni centros poblados
cols_num = ["n_ipress", "n_ipress_con_coords", "n_camas", 
            "I", "II", "III", "Sin Categoría", "n_centros_poblados"]
gdf_base[cols_num] = gdf_base[cols_num].fillna(0)

print(f"Distritos totales: {len(gdf_base):,}")
print(f"\nMuestra:")
print(gdf_base[["ubigeo", "distrito", "n_ipress", "I", "II", "III", "n_centros_poblados"]].head(5).to_string())


Distritos totales: 1,873

Muestra:
   ubigeo                distrito  n_ipress    I   II  III  n_centros_poblados
0  100902         CODO DEL POZUZO       8.0  8.0  0.0  0.0                70.0
1  100904             TOURNAVISTA       4.0  4.0  0.0  0.0                48.0
2  250305  ALEXANDER VON HUMBOLDT       2.0  2.0  0.0  0.0                 3.0
3  250302                 IRAZOLA       7.0  7.0  0.0  0.0                55.0
4  250304                 NESHUYA       8.0  8.0  0.0  0.0                 9.0


In [11]:
# Guardamos el GeoDataFrame base con todas las métricas geoespaciales
output_path = PROCESSED_DIR / "distritos_base.gpkg"
gdf_base.to_file(output_path, driver="GPKG")
print(f"Guardado en: {output_path}")
print(f"Columnas: {gdf_base.columns.tolist()}")


Guardado en: C:\Users\esarmiento\Documents\GitHub\emergency_access_peru\data\processed\distritos_base.gpkg
Columnas: ['ubigeo', 'departamen', 'provincia', 'distrito', 'area', 'geometry', 'n_ipress', 'n_ipress_con_coords', 'n_camas', 'I', 'II', 'III', 'Sin Categoría', 'n_centros_poblados']


## Resumen geoespacial

| Operación | Resultado |
|---|---|
| IPRESS asignados a distritos | 20,790 / 20,790 (100%) |
| CP con UBIGEO directo | 64,196 |
| CP asignados por spatial join | 71,528 |
| CP sin distrito (descartados) | 218 (0.16%) |
| Distritos con al menos 1 IPRESS | ver métricas |
| CRS final | EPSG:4326 |

El archivo `distritos_base.gpkg` es el input principal para `metrics.ipynb`.
